In [1]:
!python -V

Python 3.9.25


In [1]:
import os

print(os.getcwd())
print(os.path.abspath("mlflow.db"))

/home/ubuntu/mlops-zoomcamp-course/02-experiment-tracking
/home/ubuntu/mlops-zoomcamp-course/02-experiment-tracking/mlflow.db


# Q3. Train a model with autolog

In [3]:
import os
import pickle
import click
import mlflow
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("randomforest")


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


@click.command()
@click.option(
    "--data_path",
    default="./output",
    help="Location where the processed NYC taxi trip data was saved"
)
def run_train(data_path: str):
    mlflow.sklearn.autolog()
    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    with mlflow.start_run():

        rf = RandomForestRegressor(max_depth=10, random_state=0)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))


if __name__ == '__main__':
    run_train.callback(data_path="./output")

# Q5. Tune model hyperparameters

In [7]:
import os
import pickle
import click
import mlflow
import numpy as np
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("hyperparameters")


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


@click.command()
@click.option(
    "--data_path",
    default="./output",
    help="Location where the processed NYC taxi trip data was saved"
)
@click.option(
    "--num_trials",
    default=15,
    help="The number of parameter evaluations for the optimizer to explore"
)
def run_optimization(data_path: str, num_trials: int):

    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    def objective(params):

        with mlflow.start_run():
            mlflow.log_params(params)
            rf = RandomForestRegressor(**params)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))
            mlflow.log_metric("rmse", rmse)

        return {'loss': rmse, 'status': STATUS_OK}

    search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
        'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
        'random_state': 42
    }

    rstate = np.random.default_rng(42)  # for reproducible results
    fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=num_trials,
        trials=Trials(),
        rstate=rstate
    )


if __name__ == '__main__':
    run_optimization.callback(data_path="./output", num_trials=15)

  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]

🏃 View run learned-gnu-560 at: http://127.0.0.1:5000/#/experiments/2/runs/4df6fd2521704a24bfd22d34f7bec8e4

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

🏃 View run dashing-trout-593 at: http://127.0.0.1:5000/#/experiments/2/runs/b77b966ba4c64bfca519452e56da07f8

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run debonair-trout-857 at: http://127.0.0.1:5000/#/experiments/2/runs/ac9606b912c041bcbf44f13c1a56e8d9

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run bedecked-mink-755 at: http://127.0.0.1:5000/#/experiments/2/runs/9fef3c8ea713476da1e7fbad4430fa6b

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run rumbling-perch-702 at: http://127.0.0.1:5000/#/experiments/2/runs/f8e06e316cf244bb8a72975eee55d318

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

🏃 View run intrigued-goat-679 at: http://127.0.0.1:5000/#/experiments

# Q6. Promote the best model to the model registry

In [4]:
import os
import pickle
import click
import mlflow
import numpy as np
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

HPO_EXPERIMENT_NAME = "hyperparameters"
EXPERIMENT_NAME = "random-forest-best-models"
RF_PARAMS = ['max_depth', 'n_estimators', 'min_samples_split', 'min_samples_leaf', 'random_state']

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.sklearn.autolog()


def load_pickle(filename):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


def train_and_log_model(data_path, params):
    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))
    X_test, y_test = load_pickle(os.path.join(data_path, "test.pkl"))

    with mlflow.start_run():
        new_params = {}
        for param in RF_PARAMS:
            new_params[param] = int(params[param])

        rf = RandomForestRegressor(**new_params)
        rf.fit(X_train, y_train)

        # Evaluate model on the validation and test sets
        val_rmse = np.sqrt(mean_squared_error(y_val, rf.predict(X_val)))
        mlflow.log_metric("val_rmse", val_rmse)
        test_rmse = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))
        mlflow.log_metric("test_rmse", test_rmse)


@click.command()
@click.option(
    "--data_path",
    default="./output",
    help="Location where the processed NYC taxi trip data was saved"
)
@click.option(
    "--top_n",
    default=5,
    type=int,
    help="Number of top models that need to be evaluated to decide which one to promote"
)
def run_register_model(data_path: str, top_n: int):

    client = MlflowClient()

    # Retrieve the top_n model runs and log the models
    experiment = client.get_experiment_by_name(HPO_EXPERIMENT_NAME)
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=top_n,
        order_by=["metrics.rmse ASC"]
    )
    for run in runs:
        train_and_log_model(data_path=data_path, params=run.data.params)

    # Select the model with the lowest test RMSE
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    best_run = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=1,
        order_by=["metrics.test_rmse ASC"]
    )[0]

    print("Best run ID:", best_run.info.run_id)
    print("Test RMSE of best model:", best_run.data.metrics["test_rmse"])

    # Registering the model
    run_id = best_run.info.run_id
    model_uri = f"runs:/{run_id}/model"
    mlflow.register_model(model_uri, name="kunals-best-random-forest-model")


if __name__ == '__main__':
    run_register_model.callback(data_path="./output", top_n=5)

🏃 View run indecisive-fish-194 at: http://127.0.0.1:5000/#/experiments/3/runs/4e3d762d9c274e2284a9cf50504cb9df
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run victorious-trout-253 at: http://127.0.0.1:5000/#/experiments/3/runs/4f625ac3692f49dc995ef055e4bc8477
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run abrasive-mink-535 at: http://127.0.0.1:5000/#/experiments/3/runs/2b216a1e41c04281ae541b407e41c28e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run orderly-rat-743 at: http://127.0.0.1:5000/#/experiments/3/runs/3ae5faf9329a454191119dc5a341cf73
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Registered model 'kunals-best-random-forest-model' already exists. Creating a new version of this model...
2026/10/23 11:40:22 WARNING mlflow.tracking._model_registry.fluent: Run with id 4e3d762d9c274e2284a9cf50504cb9df has no artifacts at artifact path 'model', registering model based on models:/m-51778e5d9e4748b3aff7206bcda50f65 instead
2026/10/23 11:40:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: kunals-best-random-forest-model, version 2


🏃 View run suave-crane-623 at: http://127.0.0.1:5000/#/experiments/3/runs/b94d54c5dd714a38a0cd7a65738c7482
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Best run ID: 4e3d762d9c274e2284a9cf50504cb9df
Test RMSE of best model: 5.567408012462019


Created version '2' of model 'kunals-best-random-forest-model'.


In [3]:
client = mlflow.tracking.MlflowClient()

exp = client.get_experiment_by_name("hyperparameters")
print(f"Experiment ID: {exp.experiment_id}")

runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    max_results=3
)

for run in runs:
    print(run.info.run_id, run.data.metrics)

Experiment ID: 2
f7ab3870350f4331a9c81f7668b52b23 {'training_mean_squared_error': 24.20238334680275, 'training_mean_absolute_error': 3.2487582305772666, 'training_r2_score': 0.7027753287840568, 'training_root_mean_squared_error': 4.919591786602091, 'training_score': 0.7027753287840568, 'mean_squared_error_X_val': 28.676472134592508, 'rmse': 5.355041749098929}
cff0f885fcee483294c43f7046d01dcb {'training_mean_squared_error': 24.669951020257994, 'training_mean_absolute_error': 3.2760779544020098, 'training_r2_score': 0.697033222892147, 'training_root_mean_squared_error': 4.9668854446481845, 'training_score': 0.697033222892147, 'mean_squared_error_X_val': 28.85404203887249, 'rmse': 5.37159585587677}
c5614ad3d2f34920a9cf64e694b48c4c {'training_mean_squared_error': 29.87897062305349, 'training_mean_absolute_error': 3.5778548363091516, 'training_r2_score': 0.6330622859553587, 'training_root_mean_squared_error': 5.466165989343307, 'training_score': 0.6330622859553587, 'mean_squared_error_X_val